# arXiv: gravity vs mean shift

A controlled experiment on a reproducible, balanced subset of the prepared arXiv vectors. One IMSTE fit supplies a shared 2D starting layout. We then compare (1) feature-space density gravity and (2) a 2D Gaussian-kernel mean-shift update, both applied to that same layout. Labels only select a balanced sample and color the plots; neither refinement uses labels.

Run `../data/arxiv/prepare_arxiv_dataset.ipynb` first if the prepared arXiv cache is missing. Adjust `TARGET_PER_CLASS` below to control the subset size. The subset is capped at the smallest available class.


In [11]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'retina'

from pathlib import Path
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from IPython.display import display
from sklearn.neighbors import NearestNeighbors

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / 'pyproject.toml').is_file() and (path / 'imste' / '__init__.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate the IMSTE repository from the notebook working directory.')
for local_path in (PROJECT_ROOT, PROJECT_ROOT / 'notebooks' / 'src'):
    local_path = str(local_path)
    if local_path in sys.path:
        sys.path.remove(local_path)
    sys.path.insert(0, local_path)

from arxiv_dataset import load_arxiv_dataset
from arxiv_imste import select_arxiv_experiment_subset
from imste import IteratedMinimumSpanningTreeEmbedder
from imste._gravity import density_gravity_layout

DATA_DIR = PROJECT_ROOT / 'data' / 'arxiv'
CACHE_DIR = DATA_DIR / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
papers, X_full, data_manifest = load_arxiv_dataset(DATA_DIR)
print(f'Loaded {len(papers):,} papers × {X_full.shape[1]:,} features')
print(f'Vector matrix: {X_full.nbytes / 1024**3:.2f} GiB')


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Loaded 3,233,620 papers × 384 features
Vector matrix: 4.63 GiB


## 1. Select a balanced subset

`TARGET_PER_CLASS` is the number of papers per arXiv domain (the broad class labels used in notebook 05). Sampling is deterministic, without replacement, and capped by the smallest class.


In [12]:
TARGET_PER_CLASS = 1_000
SAMPLE_RANDOM_STATE = int(data_manifest['random_state'])

primary_category = papers['categories'].fillna('unknown').str.split().str[0]
archive = primary_category.str.split('.').str[0]
domain_names = {
    'cs': 'computer science', 'math': 'mathematics', 'eess': 'engineering',
    'stat': 'statistics', 'q-bio': 'biology', 'q-fin': 'finance', 'econ': 'economics',
}
domain = archive.map(domain_names).fillna('physics')
class_counts = domain.value_counts()
PER_CLASS = min(int(TARGET_PER_CLASS), int(class_counts.min()))
if PER_CLASS < TARGET_PER_CLASS:
    print(f'Capped at {PER_CLASS:,} papers per class (smallest class).')

labels_full = pd.Categorical(domain, categories=class_counts.index.tolist())
X_experiment, labels, subset_id = select_arxiv_experiment_subset(
    X_full, labels_full,
    dataset_id=data_manifest['dataset_id'],
    mode='balanced', sample_size=PER_CLASS * len(class_counts),
    random_state=SAMPLE_RANDOM_STATE,
)
X_experiment = np.asarray(X_experiment, dtype=np.float32)
label_values = np.asarray(labels).astype(str)
class_order = list(labels.categories)
print(f'Balanced subset: {len(X_experiment):,} papers ({PER_CLASS:,} per class), ID {subset_id}')
display(pd.Series(label_values).value_counts().sort_index().rename('papers').to_frame())


Balanced subset: 8,000 papers (1,000 per class), ID bfcf2a75b6f5692ac7ee


,papers
biology,1000
computer science,1000
economics,1000
engineering,1000
finance,1000
mathematics,1000
physics,1000
statistics,1000


## 2. Fit one shared IMSTE starting layout

Gravity and mean shift use identical pre-refinement coordinates. The fit disables integrated gravity so the two post-fit methods remain directly comparable. Keep this fit fixed when changing the refinement controls below.


In [13]:
RANDOM_STATE = int(data_manifest['random_state'])
IMSTE_SETTINGS = {
    'n_components': 2,
    'n_msts': 3,
    'n_epochs': 200,
    'batch_size': 32_768,
    'learning_rate': 0.05,
    'negative_ratio': 1,
    'lambda_rep': 0.2,
    'random_state': RANDOM_STATE,
    'device': 'mps',  # Change to 'auto' or 'cpu' if needed.
    'mst_method': 'famst',
    'mst_neighbor_backend': 'rinnd',
    'mst_neighbors': 15,
    'mst_inter_component_edges': 5,
    'pca_init': True,
    'pca_init_scaling': 'global',
    'multilevel_init': True,
    'multilevel_reduction_ratio': 8,
    'multilevel_base_size': 512,
    'multilevel_epochs': 10,
    'metric': 'euclidean',
    'gravity_mode': None,
    'verbose': True,
}
fit_started = time.perf_counter()
estimator = IteratedMinimumSpanningTreeEmbedder(**IMSTE_SETTINGS)
Z_base = estimator.fit_transform(X_experiment)
fit_seconds = time.perf_counter() - fit_started
Z_base = np.asarray(Z_base[:, :2], dtype=np.float32)
print(f'IMSTE starting layout: {fit_seconds:.1f} s')


Fri Nov 13 11:06:31 2026 IteratedMinimumSpanningTreeEmbedder({'batch_size': 32768, 'device': 'mps', 'epsilon': 0.0001, 'gravity_attraction_range': 1.0, 'gravity_attraction_strength': 0.5, 'gravity_core_quantile': 0.95, 'gravity_force_neighbors': 8, 'gravity_iterations': 3, 'gravity_mass_power': 0.5, 'gravity_mode': None, 'gravity_near_repulsion_strength': 2.5, 'gravity_neighbors': 15, 'gravity_pack_gap_fraction': 0.25, 'gravity_pack_radius_quantile': 0.9, 'gravity_pack_regions': False, 'gravity_repulsion_range': 0.5, 'gravity_resolution': 0.8, 'gravity_step_size': 0.05, 'gravity_strength': 0.2, 'include_knn_attraction': False, 'lambda_rep': 0.2, 'learning_rate': 0.05, 'logistic_margin': 1.0, 'logistic_temperature': 0.5, 'metric': 'euclidean', 'mst_inter_component_edges': 5, 'mst_max_neighbors': None, 'mst_method': 'famst', 'mst_neighbor_backend': 'rinnd', 'mst_neighbors': 15, 'mst_representatives_per_component': 10, 'multilevel_base_size': 512, 'multilevel_epochs': 10, 'multilevel_init

Construct MST graph:   0%|          | 0/3 [00:00<?, ?steps/s]

Building RP forest with 8 trees...
Initializing graph from 1440 leaves...
Running NN-descent for up to 13 iterations...
Iteration 1: 48537 updates
Iteration 2: 16262 updates
Iteration 3: 2256 updates
Iteration 4: 417 updates
Iteration 5: 115 updates
Converged after 5 iterations

=== Timing Breakdown ===
Forest building:      12.059ms
Leaf initialization:  13.686ms
Candidate building:    9.170ms (5 iters)
Update iterations:    25.721ms (5 iters)
Total measured:       60.636ms
Fri Nov 13 11:06:31 2026 MST graph construction complete: 0.261s


Multilevel coarsening:   0%|          | 0/8000 [00:00<?, ?nodes/s]

Multilevel refinement epochs:   0%|          | 0/30 [00:00<?, ?epoch/s]

Fri Nov 13 11:06:31 2026 Recursive initialization: refining levels
Fri Nov 13 11:06:31 2026 Recursive initialization: refining levels
Fri Nov 13 11:06:31 2026 Recursive initialization: refining levels
Fri Nov 13 11:06:31 2026 Multilevel initialization complete: 0.411s
Fri Nov 13 11:06:31 2026 Final embedding optimization


Final embedding epochs:   0%|          | 0/200 [00:00<?, ?epoch/s]

Fri Nov 13 11:06:32 2026 Final embedding optimization complete: 0.810s
Fri Nov 13 11:06:32 2026 Finished embedding
Fri Nov 13 11:06:32 2026 === Timing Breakdown ===
Fri Nov 13 11:06:32 2026 MST graph construction: 0.261s
Fri Nov 13 11:06:32 2026 Multilevel initialization: 0.411s
Fri Nov 13 11:06:32 2026 Final embedding optimization: 0.810s
Fri Nov 13 11:06:32 2026 Embedding optimization: 1.222s
Fri Nov 13 11:06:32 2026   Final MPS negative sampling and coordinate updates: 0.800s combined
Fri Nov 13 11:06:32 2026 Total measured: 1.492s
Fri Nov 13 11:06:32 2026 Multilevel initialization is included in embedding optimization.
IMSTE starting layout: 1.5 s


## 3. Cache the shared neighbor graphs

The feature-space graph is built once for gravity and reused as you change its controls. The 2D graph is also built once from `Z_base`; mean shift updates distances along these cached neighbor links, avoiding a fresh nearest-neighbor search on every slider change. Both interactive cells always start from the same fitted `Z_base`.


In [ ]:
import ipywidgets as widgets
from rinnd import RINND

MAX_GRAVITY_NEIGHBORS = 100
MAX_MEAN_SHIFT_NEIGHBORS = 100

# One feature-space ANN graph, reused across every gravity update.
feature_graph_started = time.perf_counter()
feature_search = RINND(
    np.ascontiguousarray(X_experiment, dtype=np.float32),
    metric='euclidean',
    n_neighbors=min(MAX_GRAVITY_NEIGHBORS, len(X_experiment) - 1),
    random_state=RANDOM_STATE,
    n_trees=8,
    n_iters=max(5, int(round(np.log2(len(X_experiment))))),
    max_candidates=max(20, min(60, int(round(1.5 * MAX_GRAVITY_NEIGHBORS)))),
    n_jobs=-1,
    verbose=False,
)
FEATURE_NEIGHBOR_INDICES, FEATURE_NEIGHBOR_DISTANCES = feature_search.neighbor_graph
FEATURE_NEIGHBOR_INDICES = np.asarray(FEATURE_NEIGHBOR_INDICES, dtype=np.int64)
FEATURE_NEIGHBOR_DISTANCES = np.asarray(FEATURE_NEIGHBOR_DISTANCES, dtype=np.float32)
del feature_search

# One 2D graph from the fixed starting layout. Its indices remain fixed while
# mean shift updates pair distances as the coordinates move.
mean_shift_search = NearestNeighbors(
    n_neighbors=min(MAX_MEAN_SHIFT_NEIGHBORS + 1, len(Z_base)),
    algorithm='kd_tree', n_jobs=-1,
).fit(Z_base)
MEAN_SHIFT_NEIGHBOR_DISTANCES, MEAN_SHIFT_NEIGHBOR_INDICES = mean_shift_search.kneighbors(Z_base)
_keep_nonself = MEAN_SHIFT_NEIGHBOR_INDICES != np.arange(len(Z_base), dtype=np.int64)[:, None]
MEAN_SHIFT_NEIGHBOR_INDICES = MEAN_SHIFT_NEIGHBOR_INDICES[_keep_nonself].reshape(len(Z_base), -1)
MEAN_SHIFT_NEIGHBOR_DISTANCES = MEAN_SHIFT_NEIGHBOR_DISTANCES[_keep_nonself].reshape(len(Z_base), -1)
del mean_shift_search, _keep_nonself

palette = plt.get_cmap('tab20')
color_key = {name: to_hex(palette(i % palette.N)) for i, name in enumerate(class_order)}
PLOT_COLORS = [color_key[label] for label in label_values]

def _show_refinement(title, coordinates, seconds, detail):
    coordinates = np.asarray(coordinates)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
    limits = np.vstack((Z_base, coordinates))
    low = limits.min(axis=0)
    high = limits.max(axis=0)
    center = (low + high) / 2
    radius = max(float(np.max(high - low)) * 0.52, 1e-6)
    for ax, points, panel_title in (
        (axes[0], Z_base, 'Fixed IMSTE base'),
        (axes[1], coordinates, f'{title} · {seconds:.2f} s · {detail}'),
    ):
        ax.scatter(points[:, 0], points[:, 1], c=PLOT_COLORS, s=4, alpha=0.65,
                   linewidths=0, rasterized=True)
        ax.set_title(panel_title)
        ax.set_xlim(center[0] - radius, center[0] + radius)
        ax.set_ylim(center[1] - radius, center[1] + radius)
        ax.set_aspect('equal', adjustable='box')
        ax.set_axis_off()
    display(fig)
    plt.close(fig)

print(
    f'Cached {FEATURE_NEIGHBOR_INDICES.shape[1]} feature neighbors and '
    f'{MEAN_SHIFT_NEIGHBOR_INDICES.shape[1]} 2D neighbors for {len(Z_base):,} points '
    f'in {time.perf_counter() - feature_graph_started:.2f} s.'
)


## 4. Interactive gravity

Adjust the controls to recompute gravity from the fixed IMSTE base. The feature-space neighbor search is reused; gravity updates the density regions and layout from that cached graph.


In [ ]:
def _update_gravity(mode, neighbors, resolution, strength, core_quantile,
                    mass_power, attraction_strength, near_repulsion_strength,
                    force_neighbors, step_size, iterations, pack_regions):
    global Z_gravity, gravity_seconds, gravity_diagnostics
    settings = {
        'mode': mode,
        'neighbors': int(neighbors),
        'resolution': float(resolution),
        'random_state': RANDOM_STATE,
        'core_quantile': float(core_quantile),
        'strength': float(strength),
        'mass_power': float(mass_power),
        'attraction_strength': float(attraction_strength),
        'near_repulsion_strength': float(near_repulsion_strength),
        'force_neighbors': int(force_neighbors),
        'attraction_range': 1.0,
        'repulsion_range': 0.5,
        'step_size': float(step_size),
        'iterations': int(iterations),
        'pack_regions': bool(pack_regions),
        'feature_neighbor_graph': (FEATURE_NEIGHBOR_INDICES, FEATURE_NEIGHBOR_DISTANCES),
        'feature_neighbor_metric': 'euclidean',
    }
    started = time.perf_counter()
    Z_gravity, gravity_diagnostics = density_gravity_layout(
        X_experiment, Z_base, **settings,
    )
    gravity_seconds = time.perf_counter() - started
    _show_refinement(
        'Gravity', Z_gravity, gravity_seconds,
        f"{gravity_diagnostics['region_count']} regions",
    )

_gravity_controls = {
    'mode': widgets.Dropdown(options=['density_core', 'mexican_hat'], value='density_core', description='Mode'),
    'neighbors': widgets.IntSlider(value=15, min=5, max=min(100, len(Z_base) - 1), step=5, description='Graph k', continuous_update=False),
    'resolution': widgets.FloatSlider(value=0.8, min=0.2, max=2.0, step=0.1, readout_format='.1f', description='Resolution', continuous_update=False),
    'strength': widgets.FloatSlider(value=0.5, min=0.0, max=0.9, step=0.05, readout_format='.2f', description='Pull', continuous_update=False),
    'core_quantile': widgets.FloatSlider(value=0.95, min=0.5, max=0.99, step=0.01, readout_format='.2f', description='Core q', continuous_update=False),
    'mass_power': widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.1, readout_format='.1f', description='Mass power', continuous_update=False),
    'attraction_strength': widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.1, readout_format='.1f', description='Attraction', continuous_update=False),
    'near_repulsion_strength': widgets.FloatSlider(value=1.5, min=0.0, max=3.0, step=0.1, readout_format='.1f', description='Repulsion', continuous_update=False),
    'force_neighbors': widgets.IntSlider(value=8, min=3, max=min(50, len(Z_base) - 1), step=1, description='Force k', continuous_update=False),
    'step_size': widgets.FloatSlider(value=0.05, min=0.01, max=0.2, step=0.01, readout_format='.2f', description='Step', continuous_update=False),
    'iterations': widgets.IntSlider(value=5, min=0, max=20, step=1, description='Iterations', continuous_update=False),
    'pack_regions': widgets.Checkbox(value=False, description='Pack regions'),
}
_gravity_output = widgets.interactive_output(_update_gravity, _gravity_controls)
_display_controls = [widgets.HBox([_gravity_controls['mode'], _gravity_controls['neighbors'], _gravity_controls['resolution']]),
                     widgets.HBox([_gravity_controls['strength'], _gravity_controls['core_quantile'], _gravity_controls['mass_power']]),
                     widgets.HBox([_gravity_controls['attraction_strength'], _gravity_controls['near_repulsion_strength'], _gravity_controls['force_neighbors']]),
                     widgets.HBox([_gravity_controls['step_size'], _gravity_controls['iterations'], _gravity_controls['pack_regions']])]
display(widgets.VBox(_display_controls + [_gravity_output]))


## 5. Interactive mean shift

Each update starts from `Z_base`. Neighbor identities come from its cached 2D graph; pair distances and Gaussian weights are recalculated as points move. This keeps updates responsive and avoids cumulative drift when you try different settings.


In [ ]:
def _mean_shift_from_cached_graph(*, neighbors, bandwidth_factor, step_size, iterations):
    coordinates = np.asarray(Z_base, dtype=np.float64).copy()
    neighbors = int(neighbors)
    if not 1 <= neighbors <= MEAN_SHIFT_NEIGHBOR_INDICES.shape[1]:
        raise ValueError('neighbors exceeds the cached 2D graph size.')
    bandwidth_factor = float(bandwidth_factor)
    step_size = float(step_size)
    iterations = int(iterations)
    for _ in range(iterations):
        indices = MEAN_SHIFT_NEIGHBOR_INDICES[:, :neighbors]
        offsets = coordinates[indices] - coordinates[:, None, :]
        distances = np.linalg.norm(offsets, axis=2)
        bandwidth = max(float(np.median(distances[:, -1])) * bandwidth_factor, 1e-12)
        weights = np.exp(-0.5 * (distances / bandwidth) ** 2)
        local_means = (coordinates[indices] * weights[:, :, None]).sum(axis=1)
        local_means /= np.maximum(weights.sum(axis=1, keepdims=True), 1e-12)
        coordinates += step_size * (local_means - coordinates)
        coordinates -= coordinates.mean(axis=0)
    return coordinates.astype(np.float32)


def _update_mean_shift(neighbors, bandwidth_factor, step_size, iterations):
    global Z_mean_shift, mean_shift_seconds
    started = time.perf_counter()
    Z_mean_shift = _mean_shift_from_cached_graph(
        neighbors=neighbors, bandwidth_factor=bandwidth_factor,
        step_size=step_size, iterations=iterations,
    )
    mean_shift_seconds = time.perf_counter() - started
    _show_refinement(
        'Mean shift', Z_mean_shift, mean_shift_seconds,
        f'k={neighbors}, bandwidth={bandwidth_factor:.1f}×',
    )

_mean_shift_controls = {
    'neighbors': widgets.IntSlider(value=50, min=5, max=min(100, len(Z_base) - 1), step=5, description='Neighbors', continuous_update=False),
    'bandwidth_factor': widgets.FloatSlider(value=1.0, min=0.2, max=3.0, step=0.1, readout_format='.1f', description='Bandwidth', continuous_update=False),
    'step_size': widgets.FloatSlider(value=0.25, min=0.05, max=0.75, step=0.05, readout_format='.2f', description='Step', continuous_update=False),
    'iterations': widgets.IntSlider(value=5, min=1, max=30, step=1, description='Iterations', continuous_update=False),
}
_mean_shift_output = widgets.interactive_output(_update_mean_shift, _mean_shift_controls)
display(widgets.HBox(list(_mean_shift_controls.values())))
display(_mean_shift_output)
